<a href="https://colab.research.google.com/github/Zaides01/GP2/blob/main/API_Ania.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import requests
import pandas as pd
import time

In [8]:
ACCESS_TOKEN = "vk1.a.dn7C3C24UdPqRzs7Z5daRM4LgGfCqQkVzuFcur8_cdMYCfnlnCXSJTg1y3MLmU7r4ycgG8KbfAiDaR1eHT4UjnMuwMdjS9NBmsDb_ky2LPjTOQ8MLEpUzZuML-qrGvJnMLZwT832vmhSMWIUsnbGGdT6Wxh1NBp7aOXUW1EIHHMYoP74erzHSVIlJ3IK13R8-jN4J-kF7121heIJfiv_nA"

In [26]:
GROUP_IDS = {
    '-220754053': 'VK Видео',
    '-81597813': 'Влад Бумага А4',
    '-211169870': 'ГЛЕНТ',
    '-152009330': 'ИКС',
    '-219283548': 'Вильям Бруно',
    '-147169109': 'GEO',
    '-213802301': 'ШАСТУН',
    '-195985818': 'Hardcore Fighting Championship',
    '-218565915': 'Асафьев Стас',
    '-219482354': 'Варвара Щербакова по вашим интересам!',
    '-221130436': 'Парковка',
    '-211022028': 'Арай Чобанян',
    '-218471730': 'Комьюнити',
    '-211220744': 'Стрелец-Молодец',
    '-1415705': 'GAZ',
    '-109800058': 'Medium Quality Channel'
}
VERSION = "5.199"

BASE_URL = "https://api.vk.com/method/"

def vk_request(method, params):
    url = f"{BASE_URL}{method}"
    params.update({"access_token": ACCESS_TOKEN, "v": VERSION})
    response = requests.get(url, params=params).json()
    if "error" in response:
        print(f"Ошибка: {response['error']['error_msg']}")
        return None
    return response.get("response", {})

def get_videos(group_id):
    count = 200
    offset = 0
    videos = []
    while True:
        params = {
            "owner_id": group_id,
            "count": count,
            "offset": offset,
            "extended": 1,
            "fields": "groups",
        }
        data = vk_request("video.get", params)
        items = data["items"]
        videos.extend(items)
        offset += count
        if offset >= data.get("count", 0):
            break
        time.sleep(0.3)
    return videos


all_videos = []
for group, _ in GROUP_IDS.items():
    print(f"Видео из сообщества {group}...")
    videos = get_videos(group)
    print(f"Найдено видео: {len(videos)}")
    all_videos.extend(videos)
    time.sleep(0.5)

df = pd.DataFrame(all_videos)

if "likes" in df.columns:
    df["likes_count"] = df["likes"].apply(lambda x: x.get("count")
    if isinstance(x, dict) else 0)
if "reposts" in df.columns:
    df["reposts_count"] = df["reposts"].apply(lambda x: x.get("count")
    if isinstance(x, dict) else 0)
if "image" in df.columns:
    df["image_url"] = df["image"].apply(lambda x: x[0].get("url"))

cols_to_keep = [
    "id", "owner_id", "title", "description", "duration", "date",
    "views", "comments", "likes_count", "reposts_count", "player", "can_like",
    "can_repost", "can_dislike", "is_pinned", "image_url",
]
existing_cols = [col for col in cols_to_keep if col in df.columns]
df = df[existing_cols]

df["owner_id"] = df["owner_id"].astype(str)
df["group_name"] = df["owner_id"].map(GROUP_IDS)

df.to_csv("videos_1.csv", index=False, encoding="utf-8")
print(df.shape)

Видео из сообщества -220754053...
Найдено видео: 731
Видео из сообщества -81597813...
Найдено видео: 914
Видео из сообщества -211169870...
Найдено видео: 373
Видео из сообщества -152009330...
Найдено видео: 150
Видео из сообщества -219283548...
Найдено видео: 123
Видео из сообщества -147169109...
Найдено видео: 151
Видео из сообщества -213802301...
Найдено видео: 399
Видео из сообщества -195985818...
Найдено видео: 127
Видео из сообщества -218565915...
Найдено видео: 963
Видео из сообщества -219482354...
Найдено видео: 20
Видео из сообщества -221130436...
Найдено видео: 101
Видео из сообщества -211022028...
Найдено видео: 4
Видео из сообщества -218471730...
Найдено видео: 163
Видео из сообщества -211220744...
Найдено видео: 87
Видео из сообщества -1415705...
Найдено видео: 3492
Видео из сообщества -109800058...
Найдено видео: 176
(7974, 17)
